In [ ]:
import dynamiqs as dq
dq.set_precision('double')
import numpy as np
from scipy.sparse.linalg import eigs
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from scipy.special import jv
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import matplotlib.pyplot as plt
import Plot_setting

In [ ]:
# --- constants ---
ħ = 1
e = 1
phi_quantum = 1 / 2

w_a = 25.338776456203686
w_b = 2*w_a
kappa_b = 5 / 10.4
E_J = 37*2*np.pi
phi_a = 0.11
phi_b = 0.204
epsilon_p = .1
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b 
g2 = jv(1, epsilon_p) * E_J * phi_a**2 * phi_b 

kappa_2 = 4*g**2/kappa_b
kappa_1 = 0.005*kappa_2
kappa_phi = 0

In [ ]:
n_a = 15
n_b = 2
N = n_a * n_b

# Operators
a = dq.destroy(n_a)
b = dq.destroy(n_b)
Ia = dq.eye(n_a)
Ib = dq.eye(n_b)

a_tot = dq.tensor(a, Ib)
b_tot = dq.tensor(Ia, b)

na_tot = dq.dag(a_tot) @ a_tot   # a†a
nb_tot = dq.dag(b_tot) @ b_tot   # b†b

# Nonlinear g2 term: a²b† + h.c.
term = dq.tensor(a @ a, dq.dag(b))
non_linear_op = term + dq.dag(term)

# Drive
drive_op = b_tot + dq.dag(b_tot)

# Self-Kerr and cross-Kerr operators
self_kerr_a  = na_tot @ na_tot                  # (a†a)²
self_kerr_b  = nb_tot @ nb_tot                  # (b†b)²
cross_kerr   = na_tot @ nb_tot                  # (a†a)(b†b)

La    = sp.csc_matrix((np.sqrt(kappa_1)   * a_tot).to_numpy().astype(np.complex128))
Lb    = sp.csc_matrix((np.sqrt(kappa_b)   * b_tot).to_numpy().astype(np.complex128))
L_phi = sp.csc_matrix((np.sqrt(kappa_phi) * na_tot).to_numpy().astype(np.complex128))
I     = sp.identity(N, dtype=np.complex128, format="csc")

def dissipator(L):
    LdL = L.getH() @ L
    return (
        sp.kron(L.conjugate(), L, format="csc")
        - 0.5 * sp.kron(I, LdL, format="csc")
        - 0.5 * sp.kron(LdL.T, I, format="csc")
    )

w_d = w_b

# Parameters
alpha_sq = 2.5
g2      = E_J * np.sin(epsilon_p) * phi_a**2 * phi_b
epsilon_d = 2*alpha_sq*g2
g_off   = 0*E_J * np.sin(epsilon_p)
g_off2_phia2_over_oma = g_off**2 * phi_a**2 / (w_d/2)

delta_w_a = -g_off**2*phi_a**4*(3*phi_a**2+4*phi_b**2)/(w_d/2)
delta_w_b = -4*g_off**2*phi_a**2*phi_b**2*(phi_a**2+phi_b**2)/(w_d/2)

H = (
    (w_a - w_d/2 + delta_w_a) * dq.tensor(dq.dag(a) @ a, Ib)        # Δωa a†a
  + (w_b - w_d + delta_w_b) * dq.tensor(Ia, dq.dag(b) @ b)        # Δωb b†b
  + (epsilon_d / 2) * drive_op                          # εd(b+b†)/2
  + g2 * non_linear_op                                  # g2(a²b†+h.c.)
  - g_off2_phia2_over_oma * 3 * phi_a**4 * self_kerr_a  # self-Kerr a
  - g_off2_phia2_over_oma * 4 * phi_b**4 * self_kerr_b  # self-Kerr b
  - (8 * g_off**2 * phi_a**4 * phi_b**2 / (w_d/2)) * cross_kerr  # cross-Kerr
)

H = (epsilon_d / 2) * drive_op + g2 * non_linear_op     
H = sp.csc_matrix(H.to_numpy().astype(np.complex128))

Lio = (
    -1j * (sp.kron(I, H, format="csc") - sp.kron(H.T, I, format="csc"))
    + dissipator(La)
    + dissipator(Lb)
)

In [ ]:
vals, vecs = spla.eigs(Lio, k=50, sigma=0, which="LM")
idx = np.argsort(np.real(vals))

In [ ]:
# --- Exact (dense) diagonalization ---
# The Liouvillian is only N^2 x N^2 = 900 x 900, so a full dense
# diagonalization is cheap and avoids the spurious / unconverged
# Ritz values that spla.eigs(k=50, sigma=0) returns in the spectral gap.
Lio_dense = Lio.toarray()
vals_exact = np.linalg.eigvals(Lio_dense)
vals_exact = vals_exact[np.argsort(-vals_exact.real)]  # sort by decreasing Re

print(f"# eigenvalues:            {len(vals_exact)}")
print(f"max Re(lambda):           {vals_exact.real.max():.3e}   (must be <= 0 for a physical Lindbladian)")
print(f"# with Re > 1e-9:         {np.sum(vals_exact.real > 1e-9)}")

# Conjugate-pair check: every non-real eigenvalue should have its conjugate in the spectrum
used = np.zeros(len(vals_exact), bool)
unpaired = 0
for i, v in enumerate(vals_exact):
    if used[i]:
        continue
    if abs(v.imag) < 1e-9:           # real eigenvalue
        used[i] = True
        continue
    d = np.abs(vals_exact - np.conj(v)) + np.where(used, 1e9, 0.0)
    d[i] = 1e9
    j = int(np.argmin(d))
    if d[j] < 1e-6:
        used[i] = used[j] = True
    else:
        unpaired += 1
        used[i] = True
print(f"# unpaired eigenvalues:   {unpaired}   (0 => spectrum is conjugate-symmetric)")

print("\nSlowest modes (largest Re):")
for x in vals_exact[:8]:
    print(f"  {x.real:+.6e} {x.imag:+.6e}j")

In [ ]:
vals

In [ ]:
plt.scatter(vals.real, vals.imag)
plt.xlabel(r"$|\mathrm{Re}(\lambda)|$")
plt.ylabel(r"$\mathrm{Im}(\lambda)$")
#plt.xlim(1e-4,)
plt.show()

In [ ]:
plt.scatter(np.abs(vals.real), vals.imag)
plt.xscale("log")
plt.xlabel(r"$|\mathrm{Re}(\lambda)|$")
plt.ylabel(r"$\mathrm{Im}(\lambda)$")
#plt.xlim(1e-4,)
plt.show()